# 02 Generate SOCAT Weights and DII Intermediates

Purpose: generate the SOCAT observational feature-weight and DII intermediates used by the plotting scripts.

Inputs:
- pCO2 LEAP reconstruction zarr
- SOCAT mask zarr

Outputs:
- `Finalweights_SOCAT16k_v2.txt`
- `Finalweights_SOCAT16k_v2_seed11.txt`
- `Finalweights_SOCAT16k_v2_seed12.txt`
- `Finalweights_SOCAT16k_v2_seed13.txt`
- `Finalweights_SOCAT16k_v2_seed14.txt`
- `Finalweights_SOCAT20k_v2_seed13.txt`
- `Finalimbs_SOCAT16k_v2.txt`
- `Finalimbs_SOCAT16k_v2_seed11.txt`
- `Finalimbs_SOCAT16k_v2_seed12.txt`
- `Finalimbs_SOCAT16k_v2_seed13.txt`
- `Finalimbs_SOCAT16k_v2_seed14.txt`
- `Finalimbs_SOCAT20k_v2_seed13.txt`

Estimated runtime: slow. Each job runs DADAPy backward greedy DII elimination.

Notes:
- This notebook is provenance material. This optional regeneration step requires the raw data and a compatible Python environment.
- It generates the SOCAT feature-weight and DII files used by the plotting scripts.
- Across-time SOCAT outputs belong in `03_generate_socat_across_time_weights_dii.ipynb`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from dadapy.feature_weighting import FeatureWeighting
from sklearn.preprocessing import StandardScaler

# Local paths; replace with gs:// paths if running against cloud storage.
RECONSTRUCTION_ZARR = Path("../raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr")
SOCAT_MASK_ZARR = Path("../raw/socat_mask_feb1982-dec2022.zarr")
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_SLICE = slice("2020-01-01", "2022-12-31")
N_EPOCHS = 80


## Feature Definitions

These feature definitions are used for the SOCAT feature-weighting workflow.


In [ ]:
FEATURES = [
    "sst",
    "sst_anomaly",
    "sss",
    "sss_anomaly",
    "chl_log",
    "chl_log_anomaly",
    "mld_log",
    "xco2_trend",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"


## Load SOCAT Observations

This selects SOCAT-observed points from 2020-2022 and builds the residual target `delta_fco2_1D`.


In [ ]:
def load_socat_dataframe(reconstruction_zarr, socat_mask_zarr):
    socat_mask = xr.open_dataset(socat_mask_zarr, engine="zarr")
    ds = xr.open_dataset(reconstruction_zarr, engine="zarr")
    ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))

    aligned_mask = socat_mask.reindex(time=ds.time, method="nearest")
    ds = ds.where(aligned_mask.socat_mask == 1)
    ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
    ds = ds[FEATURES + [TARGET]]
    ds = ds.sel(time=TIME_SLICE)

    return ds.to_dataframe().dropna()

SOCAT = load_socat_dataframe(RECONSTRUCTION_ZARR, SOCAT_MASK_ZARR)
SOCAT.shape


## Job List

Seed 10 uses the filename without `_seed10`; seeds 11-14 are explicit. The 20k run is retained for seed 13 to match the saved convergence inputs.


In [ ]:
JOBS = [
    (16000, 10, "SOCAT16k_v2"),
    (16000, 11, "SOCAT16k_v2_seed11"),
    (16000, 12, "SOCAT16k_v2_seed12"),
    (16000, 13, "SOCAT16k_v2_seed13"),
    (16000, 14, "SOCAT16k_v2_seed14"),
    (20000, 13, "SOCAT20k_v2_seed13"),
]


## Helper: Run One Weight/DII Job


In [ ]:
def run_weight_dii_job(socat, n_sample, seed, output_stem, n_epochs=N_EPOCHS):
    print(f"Generating {output_stem}: n={n_sample}, seed={seed}")
    sample = socat.sample(n=n_sample, random_state=seed)

    scaled = StandardScaler().fit_transform(sample.loc[:, FEATURES + [TARGET]])
    scaled = pd.DataFrame(scaled, columns=FEATURES + [TARGET])

    target = FeatureWeighting(scaled[[TARGET]].to_numpy(), verbose=True)
    inputs = FeatureWeighting(scaled[FEATURES].to_numpy(), verbose=True)

    final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
        target_data=target,
        initial_weights=None,
        n_epochs=n_epochs,
        learning_rate=None,
        decaying_lr="cos",
    )

    weights_path = OUTPUT_DIR / f"Finalweights_{output_stem}.txt"
    imbs_path = OUTPUT_DIR / f"Finalimbs_{output_stem}.txt"
    np.savetxt(weights_path, final_weights)
    np.savetxt(imbs_path, final_imbs)
    print(f"Saved {weights_path}")
    print(f"Saved {imbs_path}")


## Generate Outputs

Slow cell. Rerun only if the saved intermediates need to be recreated.


In [ ]:
for n_sample, seed, output_stem in JOBS:
    run_weight_dii_job(SOCAT, n_sample, seed, output_stem)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = []
for _, _, output_stem in JOBS:
    expected_outputs.extend([
        f"Finalweights_{output_stem}.txt",
        f"Finalimbs_{output_stem}.txt",
    ])

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
